# PDCH Text Modality — Embedding Extraction (v2)

## 0. Config & setup

In [ ]:
import shutil, os
from google.colab import drive

if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')

drive.mount('/content/drive')
print("Drive mounted")

In [ ]:
# !pip install -q torch transformers pyarrow
from pathlib import Path
import numpy as np, pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# ---- paths (point at v1 output) ----
PROC_DIR   = Path('/content/drive/MyDrive/PDCH/processed_v1')   # <-- where v1 saved
TURNS_PARQ = PROC_DIR / 'pdch_turns.parquet'
PAT_PARQ   = PROC_DIR / 'pdch_patients.parquet'
OUT_DIR    = PROC_DIR / 'embeddings'; OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- encoder ----
ENCODER_NAME = 'hfl/chinese-roberta-wwm-ext'
POOLING      = 'mean'        # 'cls' or 'mean' over token states within a chunk

# ---- chunking ----
MIN_CHARS = 4               # drop ultra-short turns
MAX_CHARS = 400            # per-chunk char budget (< 512-token cap, leaves headroom)
MAX_TOKENS = 512
BATCH = 16

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LABEL_COL_5 = 'sev5_code'; LABEL_COL_3 = 'sev3_code'
print('device:', DEVICE, '| encoder:', ENCODER_NAME)

## 1. Load v1 artifacts

In [ ]:
import glob

print("Searching for v1 output files anywhere on disk + Drive...\n")
hits = glob.glob('/content/**/pdch_turns.parquet', recursive=True) \
     + glob.glob('/content/drive/**/pdch_turns.parquet', recursive=True)
if hits:
    print("Found pdch_turns.parquet at:")
    for h in hits:
        print("   ", h)
else:
    print("NOT FOUND. v1 output is gone (runtime likely reset /content/processed).")

print("\nWhat's in your PDCH Drive folder:")
for p in sorted(glob.glob('/content/drive/MyDrive/PDCH/*')):
    print("   ", p)

In [ ]:
turns = pd.read_parquet(TURNS_PARQ)
pat   = pd.read_parquet(PAT_PARQ)
# keep only patients that survived the v1 join (have labels)
valid_ids = set(pat.patient_id)
turns = turns[turns.patient_id.isin(valid_ids)].copy()
print('patients:', pat.patient_id.nunique(), '| turns:', len(turns))
assert pat.patient_id.is_unique
pat = pat.set_index('patient_id')

## 2. Text-unit builders (the only thing that differs between A and B)

In [ ]:
def filter_short(texts, min_chars=MIN_CHARS):
    return [t for t in texts if len(t) >= min_chars]

def merge_consecutive(df_turns):
    """Collapse consecutive same-role lines into one turn. Returns list of (role, text)."""
    out = []
    for role, text in zip(df_turns.role, df_turns.text):
        if out and out[-1][0] == role:
            out[-1] = (role, out[-1][1] + ' ' + text)
        else:
            out.append((role, text))
    return out

def build_units_patient_only(g):
    """Variant A: patient turns only, in order."""
    pt = g[g.role == 'patient'].sort_values('turn_idx')
    return filter_short(pt.text.tolist())

def build_units_full_dialogue(g):
    """Variant B: merged doctor+patient, role-prefixed."""
    g = g.sort_values('turn_idx')
    merged = merge_consecutive(g)
    pref = {'patient': '患者：', 'doctor': '医生：'}
    lines = [pref[r] + t for r, t in merged if len(t) >= MIN_CHARS]
    return lines

def pack_chunks(units, max_chars=MAX_CHARS):
    """Greedy-pack units into <= max_chars chunks. Hard-split any single oversized unit."""
    chunks, cur = [], ''
    for u in units:
        if len(u) > max_chars:
            if cur: chunks.append(cur); cur = ''
            for i in range(0, len(u), max_chars):
                chunks.append(u[i:i+max_chars])
            continue
        if len(cur) + len(u) + 1 <= max_chars:
            cur = (cur + '\n' + u) if cur else u
        else:
            chunks.append(cur); cur = u
    if cur: chunks.append(cur)
    return chunks if chunks else ['']   # guard: never return empty

## 3. Encoder

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
model = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE).eval()
EMB_DIM = model.config.hidden_size
print('embedding dim:', EMB_DIM)

@torch.no_grad()
def encode_chunks(chunks):
    """List[str] -> (len(chunks), EMB_DIM) numpy."""
    vecs = []
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i:i+BATCH]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_TOKENS, return_tensors='pt').to(DEVICE)
        out = model(**enc).last_hidden_state            # (B, T, D)
        if POOLING == 'cls':
            v = out[:, 0]                               # [CLS]
        else:                                            # masked mean over tokens
            m = enc.attention_mask.unsqueeze(-1).float()
            v = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)
        vecs.append(v.cpu().numpy())
    return np.vstack(vecs)

## 4. Build per-patient embeddings for one variant

In [ ]:
def run_variant(builder, variant_name):
    ids, X, n_chunks = [], [], []
    grouped = turns.groupby('patient_id')
    for k, (pid, g) in enumerate(grouped):
        units  = builder(g)
        chunks = pack_chunks(units)
        cvecs  = encode_chunks(chunks)          # (C, D)
        pvec   = cvecs.mean(axis=0)             # mean-pool -> (D,)
        ids.append(pid); X.append(pvec); n_chunks.append(len(chunks))
        if (k+1) % 20 == 0: print(f'  [{variant_name}] {k+1}/{len(grouped)} patients')
    X = np.vstack(X).astype(np.float32)
    ids = np.array(ids)
    # align labels by id
    y5 = pat.loc[ids, LABEL_COL_5].to_numpy()
    y3 = pat.loc[ids, LABEL_COL_3].to_numpy()
    talk = pat.loc[ids, 'talk_bucket'].to_numpy()
    print(f'[{variant_name}] X={X.shape}  avg chunks/patient={np.mean(n_chunks):.1f}')
    return dict(ids=ids, X=X, y5=y5, y3=y3, talk_bucket=talk,
                n_chunks=np.array(n_chunks))

In [ ]:
variants = {
    'patient_only':  build_units_patient_only,
    'full_dialogue': build_units_full_dialogue,
}
results = {}
for name, builder in variants.items():
    print(f'\n=== {name} ===')
    results[name] = run_variant(builder, name)

## 5. Sanity checks + quick separability probe

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

def probe(res, y, label):
    X = res['X']
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000, class_weight='balanced'))
    pred = cross_val_predict(clf, X, y, cv=skf)
    return f1_score(y, pred, average='macro')

print('Linear-probe macro-F1 (5-fold patient-level CV):')
print(f"{'variant':<16}{'3-class':>10}{'5-class':>10}")
for name, res in results.items():
    f3 = probe(res, res['y3'], '3')
    f5 = probe(res, res['y5'], '5')
    print(f'{name:<16}{f3:>10.3f}{f5:>10.3f}')
print('\nNote: tiny-N probe (99 patients). Treat as a directional signal, not a final number.')

## 6. Save embeddings

In [ ]:
import json
for name, res in results.items():
    out = OUT_DIR / f'pdch_text_emb_{name}.npz'
    np.savez_compressed(out, ids=res['ids'], X=res['X'],
                        y5=res['y5'], y3=res['y3'],
                        talk_bucket=res['talk_bucket'], n_chunks=res['n_chunks'])
    print('saved', out, res['X'].shape)

meta = {
    'encoder': ENCODER_NAME, 'pooling': POOLING, 'emb_dim': int(EMB_DIM),
    'min_chars': MIN_CHARS, 'max_chars': MAX_CHARS,
    'variants': list(variants.keys()), 'n_patients': int(len(results["patient_only"]["ids"])),
}
json.dump(meta, open(OUT_DIR / 'pdch_text_emb_meta.json', 'w'), indent=2, ensure_ascii=False)
print('\n', json.dumps(meta, indent=2, ensure_ascii=False))

In [ ]:
import numpy as np
from pathlib import Path
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
import xgboost # Import xgboost

EMB_DIR = Path('/content/drive/MyDrive/PDCH/processed_v1/embeddings')

def load(variant):
    d = np.load(EMB_DIR / f'pdch_text_emb_{variant}.npz', allow_pickle=True)
    return d['X'], d['y3'].astype(int), d['y5'].astype(int)

def all_probes(X, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    models = {
        'logreg':  make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=2000, class_weight='balanced')),
        'svm_rbf': make_pipeline(StandardScaler(),
                     SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')),
        'rf':      RandomForestClassifier(n_estimators=300, class_weight='balanced',
                     random_state=42),
        'xgb':     xgboost.XGBClassifier(
                       n_estimators=300, max_depth=4, learning_rate=0.1,
                       subsample=0.8, eval_metric='mlogloss', random_state=42, use_label_encoder=False)
    }
    return {name: f1_score(y, cross_val_predict(clf, X, y, cv=skf), average='macro')
            for name, clf in models.items()}

print("Macro-F1, 5-fold patient-level CV — 3-class severity\n")
print(f"{'variant':<15}{'logreg':>9}{'svm_rbf':>9}{'rf':>9}{'xgb':>9}")
for v in ['patient_only', 'full_dialogue']:
    X, y3, y5 = load(v)
    r = all_probes(X, y3)
    print(f"{v:<15}{r['logreg']:>9.3f}{r['svm_rbf']:>9.3f}{r['rf']:>9.3f}{r['xgb']:>9.3f}")

print("\nMacro-F1, 5-fold patient-level CV — 5-class severity\n")
print(f"{'variant':<15}{'logreg':>9}{'svm_rbf':>9}{'rf':>9}{'xgb':>9}")
for v in ['patient_only', 'full_dialogue']:
    X, y3, y5 = load(v)
    r = all_probes(X, y5)
    print(f"{v:<15}{r['logreg']:>9.3f}{r['svm_rbf']:>9.3f}{r['rf']:>9.3f}{r['xgb']:>9.3f}")

In [ ]:
!pip install -q xgboost

In [ ]:
'xgb': __import__('xgboost').XGBClassifier(
                   n_estimators=300, max_depth=4, learning_rate=0.1,
                   subsample=0.8, eval_metric='mlogloss', random_state=42),